# 从图片到噪声与概率路径

In [ ]:
from pathlib import Path
import os, sys, torch
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'bs6221_flow').is_dir())
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.cache/matplotlib'))
torch.set_num_threads(4)
from bs6221_flow.sampling import load_flow, generate_images, compare_solvers, solve_ode
from bs6221_flow.training import train_model
from bs6221_flow.presentation import show_training, forward_demo, probability_paths, generation_demo, diffusion_demo, show_saved_numerics
from bs6221_flow.widgets import generation_panel, numerics_panel
from IPython.display import display
DEVICE = os.environ.get('BS6221_DEVICE', 'auto')
FLOW_WEIGHTS = None  # None: frozen 15,000-step EMA; or your training run's best.pt
model = load_flow(FLOW_WEIGHTS, DEVICE)
print('Device:', next(model.parameters()).device, '| weights SHA-256:', model.flow_weight_sha256)

## 1. 图片 → 噪声 → 概率路径 → 模型生成
先看真实训练图片逐渐加噪。Flow 插值使用固定噪声；DDPM 链每步引入新噪声。
图片到噪声记 s∈[0,1]；Flow 训练/生成记 t=1−s，噪声 t=0，图片 t=1。

In [ ]:
forward_demo(digit=3, seed=42)

### 样本空间中的构造概率路径
每张图片是 784 维向量。这里只用训练图片拟合固定 PCA 轴，投影噪声到图片的经验样本路径。
颜色表示配对的真实数字类别；这不是网络学到的生成轨迹，也不是 784 维概率密度。

In [ ]:
probability_paths(seed=42, samples=300)

### 模型实际怎样生成图片
Flow 每一步用网络预测速度，再由求解器更新状态。DDPM 预测噪声并执行反向随机采样。
下面使用新噪声，不使用前面真实图片的终点。DDPM 的“采样进度”不是 Flow 的时间参数。

In [ ]:
generation_demo(net=model, digit=3, seed=42, steps=40)
diffusion_demo(digit=3, seed=42, step_seed=43, device_preference=DEVICE)